# Execution notebook — Preprocessing pipeline 01 (Average filter)

**Type:** execution notebook.

## Purpose

Read `02_dataset/images/`, apply optional global operations and the **Average** filter, write to `02_dataset/images_pp_1/` with suffix `_PP_PL_1`.

## Imports

Loads Category A libraries from `00_common/` via `%run`.


## Configuration

Adjust values below to produce training dataset variants (image geometry unchanged).


In [ ]:
# ==========================================================
# GLOBAL OPERATIONS CONFIGURATION
# ==========================================================

APPLY_BRIGHTNESS = False
BRIGHTNESS_OFFSET = 20

APPLY_CONTRAST = False
CONTRAST_FACTOR = 1.20

APPLY_GAMMA = False
GAMMA_VALUE = 1.10

# ----------------------------------------------------------
# Pipeline identifier (do not change the pipeline number)
# ----------------------------------------------------------
PIPELINE_NUMBER = 1
PIPELINE_FILTER_NAME = "Average"

## Imports
 (`00_common`)

Loads functions from shared library notebooks — **without reimplementing algorithms**.

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.image as mpimg

# Load shared library (pedagogical order)
%run ../00_common/00_generic.ipynb
%run ../00_common/01_global_operations.ipynb
%run ../00_common/02_filtering.ipynb

## Configuration

- **Input:** `02_dataset/images`
- **Output:** `02_dataset/images_pp_1/` (consumed by segmentation)

In [ ]:
# ==========================================================
# PATHS
# ==========================================================

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
INPUT_DIR = PROJECT_ROOT / "02_dataset" / "images"
OUTPUT_DIR = PROJECT_ROOT / "02_dataset" / f"images_pp_{PIPELINE_NUMBER}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALID_IMAGE_EXTENSIONS = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")

print(f"Project root: {PROJECT_ROOT}")
print(f"Input: {INPUT_DIR}")
print(f"Output:  {OUTPUT_DIR}")

## Batch processing

For each image:
1. load and ensure `uint8` + grayscale;
2. global operations (se ativas);
3. filter **Average**;
4. save with unchanged dimensions (sem resize/crop).

In [ ]:
# ==========================================================
# GLOBAL OPERATIONS (CONFIGURÁVEIS)
# ==========================================================

def apply_configured_global_operations(input_image):
    """
    Apply only the point operations ponto a ponto ativadas na configuration section.
    """
    processed_image = input_image

    # Contrast (alpha) and brightness (beta) via library function da biblioteca comum
    if APPLY_CONTRAST or APPLY_BRIGHTNESS:
        alpha = CONTRAST_FACTOR if APPLY_CONTRAST else 1.0
        beta = BRIGHTNESS_OFFSET if APPLY_BRIGHTNESS else 0
        processed_image = apply_brightness_contrast(
            processed_image, alpha=alpha, b=beta
        )

    if APPLY_GAMMA:
        processed_image = apply_gamma_transform(processed_image, gamma=GAMMA_VALUE)

    return processed_image


# ==========================================================
# FILTRO DESTA PIPELINE
# ==========================================================

def apply_pipeline_filter(input_image):
    """Calls the function from 00_common (Average)."""
    return apply_average_filter(input_image)


# ==========================================================
# MAIN PROCESSING LOOP
# ==========================================================

input_files = sorted(
    p for p in INPUT_DIR.iterdir() if p.is_file() and p.suffix.lower() in VALID_IMAGE_EXTENSIONS
)

if len(input_files) == 0:
    print(f"WARNING: no images found em {INPUT_DIR}")
else:
    print(f"Processing {len(input_files)} image(s)...")

for input_path in input_files:
    # Step 1 — load (grayscale + uint8 via 00_common)
    original_image = load_image(input_path)
    original_height, original_width = original_image.shape

    # Step 2 — configurable global operations
    globally_processed_image = apply_configured_global_operations(original_image)

    # Step 3 — spatial filter for this pipeline
    output_image = apply_pipeline_filter(globally_processed_image)

    # Step 4 — validate geometry (compatibilidade com labels)
    output_height, output_width = output_image.shape
    if (output_height, output_width) != (original_height, original_width):
        raise ValueError(
            f"Geometry changed for {input_path.name}: "
            f"{original_height}x{original_width} -> {output_height}x{output_width}"
        )

    # Step 5 — output filename: P01_PP_PL_1.png
    original_identifier = input_path.stem
    output_filename = f"{original_identifier}_PP_PL_{PIPELINE_NUMBER}.png"
    output_path = OUTPUT_DIR / output_filename

    # Step 6 — save (intensity changes only)
    mpimg.imsave(output_path, output_image, cmap="gray")
    print(f"Guardado: {output_path.name}")

print("Pipeline 01 complete.")


## Conclusions

Pipeline 01 complete. Outputs are in `02_dataset/images_pp_1/`. Run segmentation with `DATASET_FOLDER = "images_pp_1"` when this experiment is required.
